In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. 加载数据
train = pd.read_csv('train.csv')  # 训练数据
test = pd.read_csv('test.csv')    # 测试数据


In [2]:
train['country']

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [3]:
train['days'] = pd.to_datetime(train['date'])
test['days'] = pd.to_datetime(test['date'])
base_date = pd.to_datetime('2010-01-01')
train['days'] = (train['days'] - base_date).dt.days
test['days'] = (test['days'] - base_date).dt.days
train['days'].head()

0    0
1    0
2    0
3    0
4    0
Name: days, dtype: int64

In [4]:
train['product'].describe()

count                230130
unique                    5
top       Holographic Goose
freq                  46026
Name: product, dtype: object

In [5]:
train = pd.get_dummies(train, columns=['country'], drop_first=True)  # 独热编码
test = pd.get_dummies(test, columns=['country'], drop_first=True)
train = pd.get_dummies(train, columns=['store'], drop_first=True)  # 独热编码
test = pd.get_dummies(test, columns=['store'], drop_first=True)

In [6]:
train['num_sold'] = train['num_sold'].fillna(0)

In [7]:
train.head()

,id,date,product,num_sold,days,country_Finland,country_Italy,country_Kenya,country_Norway,country_Singapore,store_Premium Sticker Mart,store_Stickers for Less
0,0,2010-01-01,Holographic Goose,0.0,0,False,False,False,False,False,False,False
1,1,2010-01-01,Kaggle,973.0,0,False,False,False,False,False,False,False
2,2,2010-01-01,Kaggle Tiers,906.0,0,False,False,False,False,False,False,False
3,3,2010-01-01,Kerneler,423.0,0,False,False,False,False,False,False,False
4,4,2010-01-01,Kerneler Dark Mode,491.0,0,False,False,False,False,False,False,False


In [8]:
features = ['days', 'country_Finland', 'country_Italy', 'country_Norway', 'country_Kenya', 'country_Singapore', 'store_Premium Sticker Mart', 'store_Stickers for Less']
X = train[features]
Y = train['num_sold']
X_test = test[features]

X_scaled = StandardScaler().fit_transform(X)
X_test_scaled = StandardScaler().fit_transform(X_test)


In [9]:
X_train, X_val, Y_train, Y_val = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)

In [10]:
svm = SVC(kernel='rbf', C=1, gamma=0.1, random_state=42)

# 模型训练
svm.fit(X_train, Y_train)

SVC(C=1, gamma=0.1, random_state=42)

In [13]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.svm import SVR  # 也可换成你的模型

# 1. 假设你已经完成模型训练，如：
# model = SVR().fit(X_train, y_train)

# 2. 验证集特征和标签
#    这里以 X_val, y_val 为例

batch_size = 1000  # 每批预测多少条，可按需调整
n_samples = X_val.shape[0]

all_preds = []  # 用于保存每个批次的预测结果

# 用 tqdm 显示分批预测进度
for start_idx in tqdm(range(0, n_samples, batch_size), desc="Predicting in batches"):
    end_idx = min(start_idx + batch_size, n_samples)
    
    # 取本批次的数据和标签
    X_batch = X_val[start_idx:end_idx]
    y_batch_true = Y_val[start_idx:end_idx]
    
    # 调用模型对本批次进行预测
    y_batch_pred = svm.predict(X_batch)
    
    # 保存预测结果
    all_preds.append(y_batch_pred)
    
    # 计算本批次的 MSE
    batch_mse = mean_squared_error(y_batch_true, y_batch_pred)
    
    # 用 tqdm.write() 打印当前批次的 MSE，避免与进度条冲突
    tqdm.write(f"Batch {start_idx // batch_size} MSE: {batch_mse:.4f}")

# 合并所有批次的预测结果
all_preds = np.concatenate(all_preds, axis=0)

# 计算整体验证集上的 MSE
final_mse = mean_squared_error(Y_val, all_preds)
print(f"\nOverall Validation MSE: {final_mse:.4f}")


Predicting in batches:   2%|▏         | 1/47 [15:42<12:02:53, 942.91s/it]

Batch 0 MSE: 754308.7820


Predicting in batches:   4%|▍         | 2/47 [30:22<11:19:15, 905.67s/it]

Batch 1 MSE: 763785.7420


Predicting in batches:   6%|▋         | 3/47 [46:25<11:23:15, 931.72s/it]

...

Batch 44 MSE: 747683.4910


Predicting in batches:  98%|█████████▊| 46/47 [10:10:35<13:11, 791.05s/it]

Batch 45 MSE: 689145.7220


Predicting in batches: 100%|██████████| 47/47 [10:10:57<00:00, 779.94s/it]

Batch 46 MSE: 1147375.4615

Overall Validation MSE: 735328.8546


In [14]:
print("支持向量：", svm.support_vectors_)


支持向量： [[-1.59860783 -0.4472136  -0.4472136  ... -0.4472136  -0.70710678
   1.41421356]
 [ 1.3859117  -0.4472136  -0.4472136  ... -0.4472136  -0.70710678
  -0.70710678]
 [-0.16257029 -0.4472136  -0.4472136  ... -0.4472136  -0.70710678
  -0.70710678]
 ...
 [ 0.24385543 -0.4472136  -0.4472136  ... -0.4472136   1.41421356
  -0.70710678]
 [ 0.7369853  -0.4472136  -0.4472136  ... -0.4472136   1.41421356
  -0.70710678]
 [-0.24927444 -0.4472136  -0.4472136  ... -0.4472136   1.41421356
  -0.70710678]]


In [15]:
print("决策函数的系数：", svm.dual_coef_)


决策函数的系数： [[ 0.  0.  0. ... -1. -1. -1.]
 [ 0.  0.  0. ... -1. -1. -1.]
 [ 0.  0.  0. ... -1. -1. -1.]
 ...
 [ 0.  0.  0. ... -1. -1. -1.]
 [ 0.  0.  0. ...  1. -1. -1.]
 [ 0.  0.  0. ...  1.  1. -1.]]
